# Notebook 08 -- Reviewer Segmentation

## 1. Load Data

In [1]:
import sys
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path
from IPython.display import display, HTML

PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

INTERIM_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Environment ready.")

Environment ready.


## 2. Reviewer Tier Distribution

In [2]:
df   = pd.read_parquet(INTERIM_DIR / "analysis_ready.parquet")
sbux = df[df["brand_category"] == "Starbucks"]

print(f"Starbucks reviews: {len(sbux):,}")
print(f"\nReviewer tiers:")
display(sbux["user_activity_tier"].value_counts().to_frame())

Starbucks reviews: 11,675

Reviewer tiers:


,count
user_activity_tier,
Casual,3778
Regular,3385
Power,2468
Elite,2044


## 3. Tier Metrics

In [3]:
tier_order = ["Casual", "Regular", "Power", "Elite"]

seg = (
    sbux.groupby("user_activity_tier")
    .agg(
        Reviews       =("review_id",      "count"),
        Avg_Stars     =("review_stars",   "mean"),
        Avg_Sentiment =("sentiment_score","mean"),
        Pct_Critical  =("star_tier",      lambda x: round((x == "Critical").mean() * 100, 1)),
        Pct_Positive  =("star_tier",      lambda x: round((x == "Positive").mean() * 100, 1)),
    )
    .reindex(tier_order)
    .round({"Avg_Stars": 2, "Avg_Sentiment": 2})
    .reset_index()
)
seg.columns = ["Tier", "Reviews", "Avg Stars", "Avg Sentiment", "% Critical", "% Positive"]
display(seg)

,Tier,Reviews,Avg Stars,Avg Sentiment,% Critical,% Positive
0,Casual,3778,2.43,0.10,64.3,30.5
1,Regular,3385,2.69,0.19,55.8,36.3
2,Power,2468,3.32,0.43,34.0,53.6
3,Elite,2044,3.60,0.56,19.6,60.9


## 4. Star Rating by Tier

In [4]:
display(seg[["Tier", "Avg Stars", "% Critical"]])

TIER_COLORS = ["#d62728", "#f4a261", "#6aaed6", "#00704A"]  # Casual→Elite

fig = go.Figure(go.Bar(
    x=seg["Tier"],
    y=seg["Avg Stars"],
    marker_color=TIER_COLORS,
    text=[f'{v:.2f}★  ({c}% critical)' for v, c in zip(seg["Avg Stars"], seg["% Critical"])],
    textposition="outside",
))
fig.add_hline(
    y=sbux["review_stars"].mean(),
    line_dash="dot", line_color="#888888",
    annotation_text=f'Starbucks avg {sbux["review_stars"].mean():.2f}',
    annotation_position="top right",
)
fig.update_layout(
    title=dict(text="Starbucks — Avg Star Rating by Reviewer Activity Tier", font=dict(size=16)),
    xaxis_title="Reviewer Tier",
    yaxis_title="Avg Star Rating",
    plot_bgcolor="white", paper_bgcolor="white",
    yaxis=dict(showgrid=True, gridcolor="#eeeeee", range=[0, 4.5]),
    xaxis=dict(showgrid=False),
    width=700, height=420,
    margin=dict(t=60, b=50, l=60, r=40),
)
fig.write_html(str(FIGURES_DIR / "08_reviewer_tier.html"))
fig.show()

,Tier,Avg Stars,% Critical
0,Casual,2.43,64.3
1,Regular,2.69,55.8
2,Power,3.32,34.0
3,Elite,3.60,19.6


## 5. Review Length by Sentiment Tier

In [5]:
tier_sent_order = ["Critical", "Neutral", "Positive"]
len_seg = (
    sbux.groupby("star_tier")
    .agg(
        Reviews    =("review_id",     "count"),
        Avg_Length =("review_length", "mean"),
        Avg_Stars  =("review_stars",  "mean"),
    )
    .reindex(tier_sent_order)
    .round(1)
    .reset_index()
)
len_seg.columns = ["Sentiment Tier", "Reviews", "Avg Length (words)", "Avg Stars"]
display(len_seg)

,Sentiment Tier,Reviews,Avg Length (words),Avg Stars
0,Critical,5557,86.4,1.3
1,Neutral,1171,91.1,3.0
2,Positive,4947,71.1,4.7


## Key Findings

- Casual reviewers average 2.43 stars with 64.3% critical reviews. Elite reviewers average 3.60 stars with 19.6% critical, a 1.17-star gap.
- Power (3.32 stars, 53.6% positive) and Elite (3.60 stars, 60.9% positive) tiers rate progressively higher.
- Critical reviews average 86.4 words vs. 71.1 for positive reviews. Dissatisfied customers write more.
- Neutral reviews are the longest at 91.1 words on average.